### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output. 

#### Pydantic

At its core:

*Pydantic is a library for defining, validating, and parsing structured data using Python classes and type hints.*

Think of it as a way to tell Python:

"This is what my data should look like. If the data is wrong, reject it or convert it."

In [1]:
## Importing a model and using it to get a response
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [2]:
from pydantic import BaseModel
from pydantic import Field

class Movie(BaseModel):
    title: str = Field(description="Title of the movie.")
    year: int = Field(description="This year the movie was released.")
    director: str = Field(description="The director of the movie.")
    rating: float = Field(description="The rating of the movie out of 10.")

In [3]:
## Now we want our model to generate structured output
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10a18a6f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10a3f3680>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [4]:
## Now first invoke the modle without the structure
model.invoke("Provide details about the movie Inception.")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie "Inception." Let me start by recalling what I know. It\'s a sci-fi action movie directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The plot involves dreams, right? People can enter others\' dreams to plant ideas. The term "inception" refers to that process.\n\nLet me think about the main characters. DiCaprio plays Dom Cobb, who\'s a professional thief who steals secrets by infiltrating the subconscious. There\'s a team he assembles for a heist. The target is a guy named Robert Fischer, whose father\'s company they want to take over. The team includes Arthur, who\'s like his right-hand man, and Eames, who can impersonate people. There\'s a new character, Ariadne, who\'s an architect and designs the dream environments. Then there\'s the projection of Cobb\'s wife, Mal, who haunts him in his dreams.\n\nThe story has layers of dreams, each deeper than the last. Time moves slower in deeper

In [ ]:
## Now, we invoke the model with structure
response = model_with_structure.invoke("Provide details about the movie Inception.")

## The response will reflect the same structure as the Movie BaseModel that we have created.
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [ ]:
## One important property of the pydantic, it will throw the error when the datatype is mismatched


In [ ]:
## Messaged output along side the parsed structure
class Movie(BaseModel):
    """A Movie with details"""
    title: str = Field(..., description="Title of the movie.")
    year: int = Field(..., description="This year the movie was released.")
    director: str = Field(..., description="The director of the movie.")
    rating: float = Field(..., description="The rating of the movie out of 10.")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

## In the response the initial raw message will also be displayed along with the parsed one.
response = model_with_structure.invoke("Provide details about the movie Interstellar")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Interstellar. Let me check the tools provided. There\'s a function called Movie that requires title, year, director, and rating. I need to make sure I have all that information for Interstellar. I know the title is "Interstellar", the director is Christopher Nolan, it came out in 2014, and the rating is probably around 8.6 based on IMDb. Let me confirm the exact rating. Yes, IMDb gives it 8.6. So I\'ll structure the tool call with these details. Make sure all required fields are included and the types are correct: title and director as strings, year as an integer, and rating as a number. No optional parameters here, so just the four required ones. Alright, that should do it.\n', 'tool_calls': [{'id': 'xc83bs0rv', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.6,"title":"Interstellar","year":2014}', 'name': 'Movie'}, 'type': 'function'}]}, re

In [7]:
## Nested Structure
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    """A Movie with details"""
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]

    ## This means the budget can be None by default
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

## In the response the initial raw message will also be displayed along with the parsed one.
response = model_with_structure.invoke("Provide details about the movie 3 Idiots")
response

MovieDetails(title='3 Idiots', year=2009, cast=[Actor(name='Aamir Khan', role='Farhan Qureshi'), Actor(name='R. Madhavan', role='Raju Rastogi'), Actor(name='Sharman Joshi', role='Advait Chandan'), Actor(name='Kareena Kapoor', role='Pia')], genre=['Comedy', 'Drama', 'Satire'], budget=2.0)

#### TypedDict

TypedDict comes from Python's typing module and is used for static type checking. TypedDict does not validate data at runtime

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year this movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of movie out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
model_with_typeddict.invoke("Please provide the details of the movie Avengers")

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [9]:
## Nested Structure
class Actor(TypedDict):
    name: Annotated[str, ..., "The name of the actor"]
    role: Annotated[str, ..., "The role they played in the movie"]

class MovieDetails(TypedDict):
    """A Movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year this movie was released"]
    cast: list[Actor]
    genre: list[str]

    ## This means the budget can be None by default
    budget: float

model_with_structure = model.with_structured_output(MovieDetails)

## In the response the initial raw message will also be displayed along with the parsed one.
response = model_with_structure.invoke("Provide details about the movie 3 Idiots")
response

{'budget': 15000000,
 'cast': [{'name': 'Aamir Khan', 'role': 'Farhan Qureshi'},
  {'name': 'R. Madhavan', 'role': 'Raju Rastogi'},
  {'name': 'Sharman Joshi', 'role': 'Chatur Ramalingam'},
  {'name': 'Boman Irani', 'role': 'Professor P. G. Ghosh'}],
 'genre': ['Comedy', 'Drama'],
 'title': '3 Idiots',
 'year': 2009}

In [10]:
## We can call the .profile attribute function on simple model, to get all the necessary information about the model
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

#### DataClasses

A data class is a class mainly containing data, although there aren't any restrictions. You create it using the @dataclass decorator

In [11]:
## First let's learn how to use the structured output with the create_agent function
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="Phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@doe.com, +919302182766"}]
})

print(result["structured_response"])

name='John Doe' email='john@doe.com' phone='+919302182766'


In [12]:
## Doing the same thing with the help of dataclass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@doe.com, +919302182766"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@doe.com', phone='+919302182766')
